# Notebook 05

## Flood Event Matching Engine

### Objective

Match historical flood events with ERA5 rainfall observations by combining spatial and temporal information to create a modelling-ready dataset.


In [1]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

SILVER_DIR = DATA_DIR / "silver"

GOLD_DIR = DATA_DIR / "gold"

print(PROJECT_ROOT)

d:\Work\Projects\India_Flood_Risk_Platform


In [3]:
# ============================================================
# Load Silver Datasets
# ============================================================

rainfall = pd.read_parquet(
    SILVER_DIR / "silver_rainfall_2023.parquet"
)

grid = pd.read_parquet(
    SILVER_DIR / "silver_grid_master.parquet"
)

events = pd.read_parquet(
    SILVER_DIR / "silver_emdat_events.parquet"
)

In [4]:
print("=" * 60)
print("Rainfall")

print(rainfall.shape)

print("=" * 60)
print("Grid")

print(grid.shape)

print("=" * 60)
print("Events")

print(events.shape)

Rainfall
(5697285, 3)
Grid
(15609, 3)
Events
(205, 24)


In [5]:
# ============================================================
# Coordinate Availability
# ============================================================

print("Latitude available :", events["latitude"].notna().sum())
print("Longitude available:", events["longitude"].notna().sum())

Latitude available : 63
Longitude available: 63


In [ ]:
# Year Distribution
events["start_year"].value_counts().sort_index()

start_year
2000     6
2001     9
2002     6
2003     6
2004     6
2005    17
2006    17
2007    16
2008     8
2009     6
2010     8
2011     7
2012     6
2013     5
2014     7
2015    10
2016     8
2017     9
2018     9
2019     5
2020     5
2021     9
2022     3
2023     5
2024     6
2025     6
Name: count, dtype: int64

In [7]:
events_2023 = events[
    events["start_year"] == 2023
].copy()

print(events_2023.shape)

events_2023[
    [
        "event_id",
        "location",
        "latitude",
        "longitude",
        "start_date"
    ]
]

(5, 24)


,event_id,location,latitude,longitude,start_date
12,2023-0428-IND,"Himachal Pradesh, Uttarakhand, Delhi, Bihar, M...",NaN,NaN,2023-06-25
13,2023-0486-IND,Mulugu District (western Telangana),NaN,NaN,2023-07-27
32,2023-0359-IND,"Mandi, Solan and Hamirpur districts (Himachal ...",NaN,NaN,2023-06-26
33,2023-0330-IND,"Nalbari, Baksa, Lakhimpur, Bajali, Barpet dist...",NaN,NaN,2023-06-14
101,2023-0846-IND,"Thoothukudi, Tirunelveli, Kanniyakumari and Te...",NaN,NaN,2023-12-18


In [8]:
# ============================================================
# Events with Available Coordinates
# ============================================================

events_geo = events.dropna(
    subset=["latitude", "longitude"]
).copy()

print("Events with coordinates :", len(events_geo))
print("Events without coordinates:", len(events) - len(events_geo))

events_geo[
    [
        "event_id",
        "latitude",
        "longitude",
        "start_date"
    ]
].head()

Events with coordinates : 63
Events without coordinates: 142


,event_id,latitude,longitude,start_date
0,2010-0416-IND,26.2350,92.25,2010-09-05
11,2016-0239-IND,27.0680,93.95,2016-06-25
17,2015-0317-IND,26.2074,82.62,2015-07-15
18,2015-0249-IND,21.8734,70.76,2015-06-19
25,2013-0282-IND,27.3870,94.92,2013-06-23


In [9]:
# ============================================================
# Prepare ERA5 Grid
# ============================================================

grid.head()

,grid_id,latitude,longitude
0,1,6.0,68.00
1,2,6.0,68.25
2,3,6.0,68.50
3,4,6.0,68.75
4,5,6.0,69.00


In [10]:
# ============================================================
# Find Nearest ERA5 Grid
# ============================================================

def nearest_grid(lat, lon, grid):

    distance = (
        (grid["latitude"] - lat) ** 2 +
        (grid["longitude"] - lon) ** 2
    )

    nearest = grid.loc[distance.idxmin()]

    return nearest["grid_id"]

In [11]:
# ============================================================
# Match Events to Nearest ERA5 Grid
# ============================================================

events_geo["grid_id"] = events_geo.apply(
    lambda row: nearest_grid(
        row["latitude"],
        row["longitude"],
        grid
    ),
    axis=1
)

events_geo.head()

,event_id,historic,disaster_type,disaster_subtype,country,location,river_basin,latitude,longitude,admin_units,...,start_day,end_year,end_month,end_day,total_deaths,people_affected,total_affected,damage_usd_000,damage_adjusted_usd_000,grid_id
0,2010-0416-IND,No,Flood,Riverine flood,India,Lakhimpur district (Assam province),Singora,26.2350,92.25,"[{""adm2_code"":70091,""adm2_name"":""Lakhimpur""}]",...,5.0,2010,9,17.0,0.0,30000.0,30000.0,NaN,NaN,9899.0
11,2016-0239-IND,No,Flood,Flash flood,India,"Chamoli, Pithoragahr districts (Uttarakhand pr...",Alaknanda and Mandakini rivers,27.0680,93.95,"[{""adm1_code"":1487,""adm1_name"":""Assam""},{""adm1...",...,25.0,2016,7,1.0,61.0,0.0,0.0,100000.0,134139.0,10269.0
17,2015-0317-IND,No,Flood,Riverine flood,India,"Kachchh, Patan, Banas Kantha districts (Gujara...",NaN,26.2074,82.62,"[{""adm1_code"":1500,""adm1_name"":""Manipur""},{""ad...",...,15.0,2015,8,19.0,293.0,13709887.0,13709887.0,NaN,NaN,9860.0
18,2015-0249-IND,No,Flood,Riverine flood,India,"Amreli, Rajkot, Surat districts (Gujarat provi...",NaN,21.8734,70.76,"[{""adm1_code"":1498,""adm1_name"":""Maharashtra""},...",...,19.0,2015,6,20.0,81.0,9000.0,9000.0,604000.0,820420.0,7635.0
25,2013-0282-IND,No,Flood,Flood (General),India,"Dhemaji, Tinsukia, Nagaon, Golaghat, Jorhat, K...",NaN,27.3870,94.92,"[{""adm2_code"":17582,""adm2_name"":""Golaghat""},{""...",...,23.0,2013,7,15.0,80.0,2000000.0,2000000.0,NaN,NaN,10515.0


In [12]:
# ============================================================
# Validate Matching
# ============================================================

print("=" * 60)
print("EVENT MATCHING SUMMARY")
print("=" * 60)

print("Events matched :", len(events_geo))
print("Missing Grid IDs :", events_geo["grid_id"].isna().sum())
print("Unique Grid IDs :", events_geo["grid_id"].nunique())

events_geo[
    [
        "event_id",
        "latitude",
        "longitude",
        "grid_id"
    ]
].head(10)

EVENT MATCHING SUMMARY
Events matched : 63
Missing Grid IDs : 0
Unique Grid IDs : 63


,event_id,latitude,longitude,grid_id
0,2010-0416-IND,26.2350,92.25,9899.0
11,2016-0239-IND,27.0680,93.95,10269.0
17,2015-0317-IND,26.2074,82.62,9860.0
18,2015-0249-IND,21.8734,70.76,7635.0
25,2013-0282-IND,27.3870,94.92,10515.0
28,2012-0334-IND,30.1135,78.85,11660.0
40,2006-0207-IND,17.4300,82.81,5626.0
41,2006-0262-IND,26.6000,94.02,10027.0
42,2006-0548-IND,32.8900,73.88,13093.0
43,2006-0549-IND,27.5900,81.52,10461.0


In [15]:
# ============================================================
# Export Event-Grid Mapping
# ============================================================

event_grid = events_geo[
    [
        "event_id",
        "grid_id",
        "latitude",
        "longitude",
        "start_date",
        "end_date"
    ]
].copy()

mapping_file = (
    SILVER_DIR /
    "silver_event_grid_mapping.parquet"
)

event_grid.to_parquet(
    mapping_file,
    index=False
)

print("Event-grid mapping exported successfully!")
print(mapping_file)

Event-grid mapping exported successfully!
d:\Work\Projects\India_Flood_Risk_Platform\data\silver\silver_event_grid_mapping.parquet


In [16]:
# ============================================================
# Engineering Summary
# ============================================================

print("=" * 60)
print("FLOOD EVENT MATCHING ENGINE")
print("=" * 60)

print(f"Total EM-DAT Events           : {len(events):,}")
print(f"Events with Coordinates       : {len(events_geo):,}")
print(f"Events Successfully Matched   : {len(event_grid):,}")
print(f"Unique ERA5 Grid Cells        : {event_grid['grid_id'].nunique():,}")

print("\nOutput")
print("-------------------------")
print("silver_event_grid_mapping.parquet")

FLOOD EVENT MATCHING ENGINE
Total EM-DAT Events           : 205
Events with Coordinates       : 63
Events Successfully Matched   : 63
Unique ERA5 Grid Cells        : 63

Output
-------------------------
silver_event_grid_mapping.parquet


In [17]:
# ============================================================
# Engineering Summary
# ============================================================

print("=" * 60)
print("FLOOD EVENT MATCHING ENGINE")
print("=" * 60)

print(f"Total EM-DAT Events           : {len(events):,}")
print(f"Events with Coordinates       : {len(events_geo):,}")
print(f"Events Successfully Matched   : {len(event_grid):,}")
print(f"Unique ERA5 Grid Cells        : {event_grid['grid_id'].nunique():,}")

print("\nOutput")
print("-------------------------")
print("silver_event_grid_mapping.parquet")

FLOOD EVENT MATCHING ENGINE
Total EM-DAT Events           : 205
Events with Coordinates       : 63
Events Successfully Matched   : 63
Unique ERA5 Grid Cells        : 63

Output
-------------------------
silver_event_grid_mapping.parquet
